Importing the Dependencies

In [45]:
import os
import cv2
import hashlib
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm 
import random
from matplotlib import pyplot as plt
from collections import Counter

Paths and a Folders

In [61]:
DATASET_PATH = os.getenv(
    
    "MHCD2022_PATH",
    "/data/UG/Kiranmoy/datasets/Military-Camouflage-MHCD2022"
    )

Dataset_Root = Path(DATASET_PATH)

IMG_DIR = Dataset_Root / "JPEGImages"
ANN_DIR = Dataset_Root / "Annotations"

SPLIT_DIR = Dataset_Root / "ImageSets" / "Main"

OUTPUT_DIR = Dataset_Root.parent / "MHCD2022_Audit"

OUTPUT_DIR.mkdir(exist_ok=True)

(OUTPUT_DIR / "reports").mkdir(exist_ok=True)
(OUTPUT_DIR / "manifests").mkdir(exist_ok=True)

FIG_DIR = OUTPUT_DIR / "figures"

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)



Class Mapping

In [47]:
CLASS_MAP = {
    "person": 0,
    "military vehicle": 1,
    "tank": 2,
    "aeroplane": 3,
    "warship": 4
}

In [48]:
split_lookup = {}

for split in ["train", "val", "test"]:

    with open(SPLIT_DIR / f"{split}.txt") as f:

        ids = [
            line.strip()
            for line in f
            if line.strip()
        ]

    for image_id in ids:
        split_lookup[image_id] = split

print("Total split entries:", len(split_lookup))

Total split entries: 3000


Create Manifest

In [49]:
manifest_rows = []

for xml_file in tqdm(list(ANN_DIR.glob("*.xml"))):

    image_id = xml_file.stem

    tree = ET.parse(xml_file)
    root = tree.getroot()

    width = int(root.find("size/width").text)
    height = int(root.find("size/height").text)

    image_area = width * height

    class_ids = []
    boxes = []
    area_ratios = []

    for obj in root.findall("object"):

        cls_name = obj.find("name").text.strip()

        if cls_name not in CLASS_MAP:
            continue

        cls_id = CLASS_MAP[cls_name]

        box = obj.find("bndbox")

        xmin = int(box.find("xmin").text)
        ymin = int(box.find("ymin").text)
        xmax = int(box.find("xmax").text)
        ymax = int(box.find("ymax").text)

        bbox_area = (xmax - xmin) * (ymax - ymin)

        class_ids.append(cls_id)

        boxes.append(
            [xmin, ymin, xmax, ymax]
        )

        area_ratios.append(
            bbox_area / image_area
        )

    manifest_rows.append({

        "image_id": image_id,

        "file_path":
        str(
            IMG_DIR / f"{image_id}.jpg"
        ),

        "split":
        split_lookup.get(
            image_id,
            "unknown"
        ),

        "width": width,
        "height": height,

        "num_objects":
        len(class_ids),

        "class_ids":
        str(class_ids),

        "box_coordinates":
        str(boxes),

        "object_area_ratios":
        str(area_ratios),

        "annotation_type":
        "bounding_box",

        "source_dataset":
        "MHCD2022",

        "license_status":
        "research",

        "notes":
        ""
    })

  0%|          | 0/3000 [00:00<?, ?it/s]

100%|██████████| 3000/3000 [00:00<00:00, 13704.79it/s]


Save Manifest

In [50]:
manifest_df = pd.DataFrame(manifest_rows)
manifest_path = OUTPUT_DIR / "manifests" / "mhcd2022_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

print(manifest_df.shape)
print(manifest_path)

(3000, 13)
/data/UG/Kiranmoy/datasets/MHCD2022_Audit/manifests/mhcd2022_manifest.csv


Dataset Summary

In [51]:
summary = {

    "total_images":
    len(manifest_df),

    "train_images":
    (manifest_df["split"]=="train").sum(),

    "val_images":
    (manifest_df["split"]=="val").sum(),

    "test_images":
    (manifest_df["split"]=="test").sum(),

    "total_objects":
    manifest_df["num_objects"].sum()
}

summary_df = pd.DataFrame(
    [summary]
)

summary_df.to_csv(
    OUTPUT_DIR
    / "reports"
    / "dataset_summary.csv",
    index=False
)

summary_df

,total_images,train_images,val_images,test_images,total_objects
0,3000,1920,480,600,4401


Class-Wise Instance Count

In [52]:
Class_counts = {
     k : 0
     for k in CLASS_MAP.keys()
}
for xml_file in tqdm(list(ANN_DIR.glob("*.xml"))):

    tree = ET.parse(xml_file)
    root = tree.getroot()

    for obj in root.findall("object"):

        cls_name = obj.find("name").text.strip()

        if cls_name not in CLASS_MAP:
            continue

        Class_counts[cls_name] += 1

class_df = pd.DataFrame({
    "class": Class_counts.keys(),
    "isinstance_count": Class_counts.values()
})

class_df.to_csv(
    OUTPUT_DIR
    / "reports"
    / "class_summary.csv",
    index=False
)
class_df

  0%|          | 0/3000 [00:00<?, ?it/s]

100%|██████████| 3000/3000 [00:00<00:00, 22260.28it/s]


,class,isinstance_count
0,person,3364
1,military vehicle,198
2,tank,399
3,aeroplane,286
4,warship,154


Resolution Report

In [53]:
resolution_df = manifest_df[
    ["width", "height"]
]

resolution_df.to_csv(
    OUTPUT_DIR
    / "reports"
    / "resolution_distribution.csv",
    index=False
)

resolution_df.describe()

,width,height
count,3000.000000,3000.000000
mean,714.098000,434.340333
std,194.946024,155.992227
min,200.000000,146.000000
25%,596.000000,336.000000
50%,640.000000,426.000000
75%,854.000000,480.000000
max,2810.000000,1880.000000


Object Size Distribution

In [54]:
size_rows = []

for xml_file in tqdm(list(ANN_DIR.glob("*.xml"))):

    tree = ET.parse(xml_file)
    root = tree.getroot()

    width = int(root.find("size/width").text)
    height = int(root.find("size/height").text)

    image_area = width * height

    for obj in root.findall("object"):

        box = obj.find("bndbox")

        xmin = int(box.find("xmin").text)
        ymin = int(box.find("ymin").text)
        xmax = int(box.find("xmax").text)
        ymax = int(box.find("ymax").text)

        area = (
            (xmax - xmin)
            *
            (ymax - ymin)
        )

        ratio = area / image_area

        if ratio < 0.01:
            size = "small"

        elif ratio < 0.10:
            size = "medium"

        else:
            size = "large"

        size_rows.append({
            "ratio": ratio,
            "size": size
        })

size_df = pd.DataFrame(size_rows)

size_df.to_csv(
    OUTPUT_DIR
    / "reports"
    / "object_size_distribution.csv",
    index=False
)

size_df["size"].value_counts()

100%|██████████| 3000/3000 [00:00<00:00, 20246.46it/s]


size
medium    1692
small     1656
large     1053
Name: count, dtype: int64

Missing Annotation Report

In [55]:
missing = []

for img_file in IMG_DIR.glob("*.jpg"):

    xml_file = (
        ANN_DIR
        / f"{img_file.stem}.xml"
    )

    if not xml_file.exists():

        missing.append(
            img_file.stem
        )

missing_df = pd.DataFrame({
    "image_id": missing
})

missing_df.to_csv(
    OUTPUT_DIR
    / "reports"
    / "missing_annotation_report.csv",
    index=False
)

print(
    "Missing:",
    len(missing_df)
)

Missing: 0


Train/Test Leakage Check 

In [56]:
train_ids = set(
    pd.read_csv(
        SPLIT_DIR/"train.txt",
        header=None
    )[0]
)

test_ids = set(
    pd.read_csv(
        SPLIT_DIR/"test.txt",
        header=None
    )[0]
)

overlap = train_ids.intersection(
    test_ids
)

pd.DataFrame({
    "overlap":
    list(overlap)
}).to_csv(
    OUTPUT_DIR
    / "reports"
    / "train_test_leakage_report.csv",
    index=False
)

print(
    "Leakage:",
    len(overlap)
)

Leakage: 0


All The Required Figures in Single Pipeline

In [57]:
# Figure 1 : Sample Label Visualisation

random.seed(42)

samples = random.sample(
    list(ANN_DIR.glob("*.xml")),
    9
)

fig, axes = plt.subplots(
    3,
    3,
    figsize=(15,15)
)

for ax, xml_file in zip(
    axes.flatten(),
    samples
):

    image_id = xml_file.stem

    img = cv2.imread(
        str(
            IMG_DIR /
            f"{image_id}.jpg"
        )
    )

    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    tree = ET.parse(xml_file)
    root = tree.getroot()

    for obj in root.findall("object"):

        cls = obj.find("name").text

        box = obj.find("bndbox")

        xmin = int(box.find("xmin").text)
        ymin = int(box.find("ymin").text)
        xmax = int(box.find("xmax").text)
        ymax = int(box.find("ymax").text)

        cv2.rectangle(
            img,
            (xmin,ymin),
            (xmax,ymax),
            (0,255,0),
            2
        )

        cv2.putText(
            img,
            cls,
            (xmin,max(20,ymin)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255,0,0),
            2
        )

    ax.imshow(img)
    ax.axis("off")

plt.tight_layout()

plt.savefig(
    FIG_DIR /
    "sample_label_visualisation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("✓ sample_label_visualisation.png")


# Figure 2 : Class Distribution

class_counts = {
    "person":0,
    "military vehicle":0,
    "tank":0,
    "aeroplane":0,
    "warship":0
}

for xml_file in tqdm(
    ANN_DIR.glob("*.xml"),
    desc="Counting classes"
):

    tree = ET.parse(xml_file)
    root = tree.getroot()

    for obj in root.findall("object"):

        cls = obj.find("name").text.strip()

        if cls in class_counts:
            class_counts[cls] += 1

class_df = pd.DataFrame({

    "Class":
    class_counts.keys(),

    "Count":
    class_counts.values()

})

plt.figure(figsize=(8,5))

plt.bar(
    class_df["Class"],
    class_df["Count"]
)

plt.xticks(rotation=20)

plt.ylabel("Instances")

plt.title(
    "MHCD2022 Class Distribution"
)

plt.tight_layout()

plt.savefig(
    FIG_DIR /
    "class_distribution.png",
    dpi=300
)

plt.close()

print("✓ class_distribution.png")


# Figure 3 : Object Size Histogram

ratios = []

for xml_file in tqdm(
    ANN_DIR.glob("*.xml"),
    desc="Computing object sizes"
):

    tree = ET.parse(xml_file)
    root = tree.getroot()

    width = int(
        root.find("size/width").text
    )

    height = int(
        root.find("size/height").text
    )

    image_area = width * height

    for obj in root.findall("object"):

        box = obj.find("bndbox")

        xmin = int(box.find("xmin").text)
        ymin = int(box.find("ymin").text)
        xmax = int(box.find("xmax").text)
        ymax = int(box.find("ymax").text)

        area = (
            (xmax - xmin)
            *
            (ymax - ymin)
        )

        ratios.append(
            area / image_area
        )

plt.figure(figsize=(8,5))

plt.hist(
    ratios,
    bins=50
)

plt.xlabel(
    "Object Area Ratio"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "Object Size Distribution"
)

plt.tight_layout()

plt.savefig(
    FIG_DIR /
    "object_size_histogram.png",
    dpi=300
)

plt.close()

print("✓ object_size_histogram.png")


# Figure 4 : Small Object Examples Grid

small_images = []

for xml_file in ANN_DIR.glob("*.xml"):

    tree = ET.parse(xml_file)
    root = tree.getroot()

    width = int(
        root.find("size/width").text
    )

    height = int(
        root.find("size/height").text
    )

    image_area = width * height

    found = False

    for obj in root.findall("object"):

        box = obj.find("bndbox")

        xmin = int(box.find("xmin").text)
        ymin = int(box.find("ymin").text)
        xmax = int(box.find("xmax").text)
        ymax = int(box.find("ymax").text)

        area = (
            (xmax-xmin)
            *
            (ymax-ymin)
        )

        ratio = area / image_area

        if ratio < 0.01:
            found = True

    if found:
        small_images.append(
            xml_file.stem
        )

small_images = small_images[:9]

fig, axes = plt.subplots(
    3,
    3,
    figsize=(15,15)
)

for ax, image_id in zip(
    axes.flatten(),
    small_images
):

    img = cv2.imread(
        str(
            IMG_DIR /
            f"{image_id}.jpg"
        )
    )

    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    ax.imshow(img)

    ax.set_title(
        "Small Object"
    )

    ax.axis("off")

plt.tight_layout()

plt.savefig(
    FIG_DIR /
    "difficulty_example_grid.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("✓ difficulty_example_grid.png")

# Done

print("\nAll figures generated successfully.")
print(FIG_DIR)

✓ sample_label_visualisation.png


Counting classes: 3000it [00:00, 40741.18it/s]


✓ class_distribution.png


Computing object sizes: 3000it [00:00, 33383.42it/s]


✓ object_size_histogram.png
✓ difficulty_example_grid.png

All figures generated successfully.
/data/UG/Kiranmoy/datasets/MHCD2022_Audit/figures
